In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [2]:
df = pd.read_csv('../../datasets/c_text_youm7/youm7_nlp_preprocessed_data.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12089 entries, 0 to 12088
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   category_label   12089 non-null  str  
 1   summary_no_stem  12089 non-null  str  
 2   summary_stemmed  12089 non-null  str  
dtypes: str(3)
memory usage: 5.2 MB


# Stemmed Training

In [4]:
df_stemmed = df[['category_label', 'summary_stemmed']].copy()

In [5]:
df_stemmed.info()

<class 'pandas.DataFrame'>
RangeIndex: 12089 entries, 0 to 12088
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   category_label   12089 non-null  str  
 1   summary_stemmed  12089 non-null  str  
dtypes: str(2)
memory usage: 2.2 MB


In [6]:
# 70% Train, 30% Temp (Dev + Test)
X_train, X_temp, y_train, y_temp = train_test_split(
    df_stemmed["summary_stemmed"],
    df_stemmed["category_label"],
    test_size=0.30,
    random_state=42,
    stratify=df_stemmed["category_label"]
)

# 15% Dev, 15% Test
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [7]:
best_c = None
best_dev_f1 = -1
C_values = [0.001, 0.01, 0.1, 1, 2, 5, 10, 20, 50, 100, 500, 1000, 2000, 5000, 10000, 20000, 50000]

In [8]:
results = []

for c in C_values:

    candidate = make_pipeline(
        TfidfVectorizer(
            ngram_range=(1, 2),
            norm="l2",
            sublinear_tf=True
        ),
        LinearSVC(
            C=c,
            random_state=42
        )
    )

    candidate.fit(X_train, y_train)

    # Training predictions
    train_preds = candidate.predict(X_train)

    # Development predictions
    dev_preds = candidate.predict(X_dev)

    train_f1 = f1_score(
        y_train,
        train_preds,
        average="macro"
    )

    dev_f1 = f1_score(
        y_dev,
        dev_preds,
        average="macro"
    )

    dev_acc = accuracy_score(y_dev, dev_preds)

    results.append({
        "C": c,
        "Train Macro F1": train_f1,
        "Dev Macro F1": dev_f1,
        "Dev Accuracy": dev_acc
    })

    if dev_f1 > best_dev_f1:
        best_dev_f1 = dev_f1
        best_c = c


results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))
print(f"\nBest C: {best_c}")
print(f"Best Dev Macro F1: {best_dev_f1:.4f}")

/home/m7ms/Downloads/machine-learning-models-2026-fcai-cu/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/m7ms/Downloads/machine-learning-models-2026-fcai-cu/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/m7ms/Downloads/machine-learning-models-2026-fcai-cu/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/m7ms/Downloads/machine-learning-models-2026-fcai-cu/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/m7ms/Downloads/machine-learning-models-2026-fcai-cu/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:129

        C  Train Macro F1  Dev Macro F1  Dev Accuracy
    0.001        0.766073      0.770974      0.767788
    0.010        0.906157      0.884451      0.884721
    0.100        0.969257      0.914538      0.914506
    1.000        0.999874      0.922789      0.922780
    2.000        0.999874      0.924900      0.924986
    5.000        1.000000      0.924201      0.924435
   10.000        1.000000      0.924211      0.924435
   20.000        1.000000      0.924750      0.924986
   50.000        1.000000      0.924243      0.924435
  100.000        1.000000      0.924243      0.924435
  500.000        1.000000      0.924243      0.924435
 1000.000        1.000000      0.924243      0.924435
 2000.000        1.000000      0.924243      0.924435
 5000.000        1.000000      0.924243      0.924435
10000.000        1.000000      0.924243      0.924435
20000.000        1.000000      0.924243      0.924435
50000.000        1.000000      0.924243      0.924435

Best C: 2
Best Dev Macro F1

In [9]:
X_train_final = pd.concat([X_train, X_dev])
y_train_final = pd.concat([y_train, y_dev])

In [10]:
stemmed_svm_model = make_pipeline(
    TfidfVectorizer(
        ngram_range=(1, 2),
        norm="l2",
        sublinear_tf=True
    ),
    LinearSVC(
        C=best_c,
        random_state=42
    )
)

In [11]:
stemmed_svm_model.fit(
    X_train_final,
    y_train_final
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('linearsvc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](7,)","['اقتصاد_وبورصة','حوادث_وقضايا','رياضة_مصرية',...,'علوم_وتكنولوجيا', 'فن_وثقافة','محافظات_ومجتمع']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [12]:
test_preds = stemmed_svm_model.predict(X_test)

print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(f"Test Macro F1: " f"{f1_score(y_test, test_preds, average='macro'):.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_preds,
        zero_division=0
    )
)

Test Accuracy: 0.9350
Test Macro F1: 0.9353

Classification Report:
                  precision    recall  f1-score   support

   اقتصاد_وبورصة       0.95      0.95      0.95       261
    حوادث_وقضايا       0.93      0.95      0.94       223
     رياضة_مصرية       0.98      0.97      0.98       252
سياسة_وأخبار_مصر       0.94      0.87      0.90       268
 علوم_وتكنولوجيا       0.93      0.97      0.95       282
       فن_وثقافة       0.94      0.91      0.93       262
  محافظات_ومجتمع       0.88      0.93      0.91       266

        accuracy                           0.93      1814
       macro avg       0.94      0.94      0.94      1814
    weighted avg       0.94      0.93      0.93      1814



# Testing valid across both stemming, non stemmed

In [13]:
def get_classification_results(
    y_true,
    y_pred,
    decision_scores,
    classes
):

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    decision_scores = np.asarray(decision_scores)

    # Classification report
    report = classification_report(
        y_true,
        y_pred,
        labels=classes,
        output_dict=True,
        zero_division=0
    )

    # Confusion matrix
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=classes
    )

    rows = []

    # Per-class metrics
    for i, label in enumerate(classes):

        metrics = report[str(label)]

        # One-vs-Rest binary classification
        y_binary = (y_true == label).astype(int)

        # True Positives, False Positives,
        # True Negatives, False Negatives
        tp = cm[i, i]
        fp = cm[:, i].sum() - tp
        fn = cm[i, :].sum() - tp
        tn = cm.sum() - (tp + fp + fn)

        # Specificity
        specificity = (
            tn / (tn + fp)
            if (tn + fp) > 0 else np.nan
        )

        # ROC-AUC and PR-AUC
        if len(np.unique(y_binary)) == 2:

            roc_auc = roc_auc_score(
                y_binary,
                decision_scores[:, i]
            )

            pr_auc = average_precision_score(
                y_binary,
                decision_scores[:, i]
            )

        else:
            roc_auc = np.nan
            pr_auc = np.nan

        rows.append({
            "Category": label,
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1-score": metrics["f1-score"],
            "Specificity": specificity,
            "ROC-AUC": roc_auc,
            "PR-AUC (AP)": pr_auc,
            "Accuracy": np.nan,
            "Support": int(metrics["support"])
        })

    # Create per-class DataFrame
    results_df = pd.DataFrame(rows)

    # Macro Average
    macro_row = {
        "Category": "Macro Average",
        "Precision": report["macro avg"]["precision"],
        "Recall": report["macro avg"]["recall"],
        "F1-score": report["macro avg"]["f1-score"],
        "Specificity": results_df["Specificity"].mean(),
        "ROC-AUC": results_df["ROC-AUC"].mean(),
        "PR-AUC (AP)": results_df["PR-AUC (AP)"].mean(),
        "Accuracy": np.nan,
        "Support": len(y_true)
    }

    # Weighted Average
    weights = results_df["Support"].to_numpy()

    weighted_row = {
        "Category": "Weighted Average",
        "Precision": report["weighted avg"]["precision"],
        "Recall": report["weighted avg"]["recall"],
        "F1-score": report["weighted avg"]["f1-score"],
        "Specificity": np.average(
            results_df["Specificity"],
            weights=weights
        ),
        "ROC-AUC": np.average(
            results_df["ROC-AUC"],
            weights=weights
        ),
        "PR-AUC (AP)": np.average(
            results_df["PR-AUC (AP)"],
            weights=weights
        ),
        "Accuracy": np.nan,
        "Support": len(y_true)
    }

    # Overall Accuracy
    accuracy_row = {
        "Category": "Overall Accuracy",
        "Precision": np.nan,
        "Recall": np.nan,
        "F1-score": np.nan,
        "Specificity": np.nan,
        "ROC-AUC": np.nan,
        "PR-AUC (AP)": np.nan,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Support": len(y_true)
    }

    # Combine all rows
    results_df = pd.concat(
        [
            results_df,
            pd.DataFrame([
                macro_row,
                weighted_row,
                accuracy_row
            ])
        ],
        ignore_index=True
    )

    return results_df

# Testing the Stemmed Svm

In [14]:
test_scores = stemmed_svm_model.decision_function(X_test)

In [15]:
stemmed_results_table = get_classification_results(
    y_test,
    test_preds,
    test_scores,
    stemmed_svm_model.classes_
)

In [16]:
display(
    stemmed_results_table.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "Specificity": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC (AP)": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Support": "{:,.0f}"
    }, na_rep="—")
)

,Category,Precision,Recall,F1-score,Specificity,ROC-AUC,PR-AUC (AP),Accuracy,Support
0,اقتصاد_وبورصة,0.9466,0.9502,0.9484,0.9910,0.9924,0.9775,—,261
1,حوادث_وقضايا,0.9298,0.9507,0.9401,0.9899,0.9966,0.9713,—,223
2,رياضة_مصرية,0.9839,0.9683,0.9760,0.9974,0.9992,0.9966,—,252
3,سياسة_وأخبار_مصر,0.9431,0.8657,0.9027,0.9909,0.9848,0.9532,—,268
4,علوم_وتكنولوجيا,0.9254,0.9681,0.9463,0.9856,0.9972,0.9835,—,282
5,فن_وثقافة,0.9447,0.9122,0.9282,0.9910,0.9915,0.9696,—,262
6,محافظات_ومجتمع,0.8794,0.9323,0.9051,0.9780,0.9879,0.9468,—,266
7,Macro Average,0.9361,0.9353,0.9353,0.9891,0.9928,0.9712,—,"1,814"
8,Weighted Average,0.9358,0.9350,0.9349,0.9890,0.9927,0.9711,—,"1,814"
9,Overall Accuracy,—,—,—,—,—,—,0.9350,"1,814"


# No Stem Training

In [17]:
df_no_stem = df[['category_label', 'summary_no_stem']].copy()
df_no_stem.info()

<class 'pandas.DataFrame'>
RangeIndex: 12089 entries, 0 to 12088
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   category_label   12089 non-null  str  
 1   summary_no_stem  12089 non-null  str  
dtypes: str(2)
memory usage: 3.4 MB


In [18]:
X_train_no_stem, X_temp_no_stem, y_train_no_stem, y_temp_no_stem = train_test_split(
    df_no_stem["summary_no_stem"],
    df_no_stem["category_label"],
    test_size=0.30,
    random_state=42,
    stratify=df_no_stem["category_label"]
)

X_dev_no_stem, X_test_no_stem, y_dev_no_stem, y_test_no_stem = train_test_split(
    X_temp_no_stem,
    y_temp_no_stem,
    test_size=0.50,
    random_state=42,
    stratify=y_temp_no_stem
)

In [19]:
best_c_no_stem = None
best_dev_f1_no_stem = -1
C_values = [0.001, 0.01, 0.1, 1, 2, 5, 10, 20, 50, 100, 500, 1000, 2000, 5000, 10000, 20000, 50000]

In [20]:
results = []

for c in C_values:

    candidate = make_pipeline(
        TfidfVectorizer(
            ngram_range=(1, 2),
            norm="l2",
            sublinear_tf=True
        ),
        LinearSVC(
            C=c,
            random_state=42
        )
    )

    candidate.fit(X_train_no_stem, y_train_no_stem)

    # Training predictions
    train_preds = candidate.predict(X_train_no_stem)

    # Development predictions
    dev_preds = candidate.predict(X_dev_no_stem)

    train_f1 = f1_score(
        y_train_no_stem,
        train_preds,
        average="macro"
    )

    dev_f1 = f1_score(
        y_dev_no_stem,
        dev_preds,
        average="macro"
    )

    dev_acc = accuracy_score(y_dev_no_stem, dev_preds)

    results.append({
        "C": c,
        "Train Macro F1": train_f1,
        "Dev Macro F1": dev_f1,
        "Dev Accuracy": dev_acc
    })
    
    if dev_f1 > best_dev_f1_no_stem:
        best_dev_f1_no_stem = dev_f1
        best_c_no_stem = c


results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))
print(f"\nBest C: {best_c_no_stem}")
print(f"Best Dev Macro F1: {best_dev_f1_no_stem:.4f}")

/home/m7ms/Downloads/machine-learning-models-2026-fcai-cu/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/m7ms/Downloads/machine-learning-models-2026-fcai-cu/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/m7ms/Downloads/machine-learning-models-2026-fcai-cu/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/m7ms/Downloads/machine-learning-models-2026-fcai-cu/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:1295: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/m7ms/Downloads/machine-learning-models-2026-fcai-cu/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:129

        C  Train Macro F1  Dev Macro F1  Dev Accuracy
    0.001        0.664769      0.695087      0.685052
    0.010        0.916797      0.874762      0.873690
    0.100        0.980054      0.905845      0.905681
    1.000        0.999874      0.924748      0.924435
    2.000        0.999874      0.925179      0.924986
    5.000        1.000000      0.928417      0.928296
   10.000        1.000000      0.927819      0.927744
   20.000        1.000000      0.927826      0.927744
   50.000        1.000000      0.927826      0.927744
  100.000        1.000000      0.927826      0.927744
  500.000        1.000000      0.927826      0.927744
 1000.000        1.000000      0.927826      0.927744
 2000.000        1.000000      0.927826      0.927744
 5000.000        1.000000      0.927826      0.927744
10000.000        1.000000      0.927826      0.927744
20000.000        1.000000      0.927826      0.927744
50000.000        1.000000      0.927826      0.927744

Best C: 5
Best Dev Macro F1

In [21]:
X_train_final_no_stem = pd.concat([X_train_no_stem, X_dev_no_stem])
y_train_final_no_stem= pd.concat([y_train_no_stem, y_dev_no_stem])

In [22]:
no_stem_svm_model = make_pipeline(
    TfidfVectorizer(
        ngram_range=(1, 2),
        norm="l2",
        sublinear_tf=True
    ),
    LinearSVC(
        C=best_c_no_stem,
        random_state=42
    )
)

In [23]:
no_stem_svm_model.fit(
    X_train_final_no_stem,
    y_train_final_no_stem
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('linearsvc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](7,)","['اقتصاد_وبورصة','حوادث_وقضايا','رياضة_مصرية',...,'علوم_وتكنولوجيا', 'فن_وثقافة','محافظات_ومجتمع']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [24]:
test_preds = no_stem_svm_model.predict(X_test_no_stem)

print(f"Test Accuracy: {accuracy_score(y_test_no_stem, test_preds):.4f}")
print(f"Test Macro F1: " f"{f1_score(y_test_no_stem, test_preds, average='macro'):.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_test_no_stem,
        test_preds,
        zero_division=0
    )
)

Test Accuracy: 0.9427
Test Macro F1: 0.9429

Classification Report:
                  precision    recall  f1-score   support

   اقتصاد_وبورصة       0.95      0.95      0.95       261
    حوادث_وقضايا       0.93      0.96      0.94       223
     رياضة_مصرية       0.99      0.99      0.99       252
سياسة_وأخبار_مصر       0.95      0.87      0.91       268
 علوم_وتكنولوجيا       0.92      0.98      0.95       282
       فن_وثقافة       0.96      0.93      0.94       262
  محافظات_ومجتمع       0.90      0.93      0.91       266

        accuracy                           0.94      1814
       macro avg       0.94      0.94      0.94      1814
    weighted avg       0.94      0.94      0.94      1814



# Testing No Stem Svm

In [25]:
test_scores = no_stem_svm_model.decision_function(X_test_no_stem)

In [26]:
no_stem_results_table = get_classification_results(
    y_test_no_stem,
    test_preds,
    test_scores,
    no_stem_svm_model.classes_
)

In [27]:
display(
    no_stem_results_table.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-score": "{:.4f}",
        "Specificity": "{:.4f}",
        "ROC-AUC": "{:.4f}",
        "PR-AUC (AP)": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Support": "{:,.0f}"
    }, na_rep="—")
)

,Category,Precision,Recall,F1-score,Specificity,ROC-AUC,PR-AUC (AP),Accuracy,Support
0,اقتصاد_وبورصة,0.9537,0.9464,0.9500,0.9923,0.9950,0.9807,—,261
1,حوادث_وقضايا,0.9304,0.9596,0.9448,0.9899,0.9954,0.9621,—,223
2,رياضة_مصرية,0.9881,0.9921,0.9901,0.9981,0.9997,0.9982,—,252
3,سياسة_وأخبار_مصر,0.9547,0.8657,0.9080,0.9929,0.9826,0.9517,—,268
4,علوم_وتكنولوجيا,0.9200,0.9787,0.9485,0.9843,0.9937,0.9752,—,282
5,فن_وثقافة,0.9569,0.9313,0.9439,0.9929,0.9943,0.9810,—,262
6,محافظات_ومجتمع,0.9015,0.9286,0.9148,0.9826,0.9861,0.9388,—,266
7,Macro Average,0.9436,0.9432,0.9429,0.9904,0.9924,0.9697,—,"1,814"
8,Weighted Average,0.9433,0.9427,0.9425,0.9903,0.9923,0.9696,—,"1,814"
9,Overall Accuracy,—,—,—,—,—,—,0.9427,"1,814"
